### Carregando Dataset Original

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('../Datasets/Dados/VIF - Final até agora/dataset_modelo_2022.csv', sep=';')
df.head()

,NU_ANO_CENSO,CO_MUNICIPIO,NO_MUNICIPIO,SG_UF,TAXA_CRE_INT,PROP_ESC_PUBLICA,PROP_ESC_URBANA,PROP_MAT_BRANCA,PROP_MAT_PRETA,PROP_MAT_PARDA,...,TAXA_OCUP_FEM,TAXA_OCUP_MASC,PROP_MAES_JOVENS,PROP_DOM_SEM_CONJUGE_MULHER,PERC_RENDA_OUTRAS_FONTES,RENDA_MEDIANA_RESPONSAVEL,IDH_PROP_POBREZA_CRIANCAS,IDH_GINI,IDH_EXPECT_ANOS_ESTUDO,IDH_RAZAO_DEPENDENCIA
0,2022,2100055,Açailândia,MA,0.135303,0.817391,0.695652,0.145048,0.016151,0.836538,...,0.374363,0.631811,0.014926,0.151264,21.47,1212.0,36.65,0.56,9.34,55.43
1,2022,2100105,Afonso Cunha,MA,0.864198,1.000000,0.240000,0.040602,0.023903,0.932875,...,0.275815,0.419239,0.037771,0.124486,43.42,1200.0,78.75,0.59,8.62,74.96
2,2022,2100154,Água Doce do Maranhão,MA,0.000000,0.969697,0.242424,0.090790,0.014749,0.890528,...,0.311589,0.473313,0.022701,0.110829,45.26,1200.0,71.85,0.59,8.07,65.18
3,2022,2100204,Alcântara,MA,0.000000,0.982143,0.142857,0.066054,0.067747,0.865231,...,0.261981,0.421794,0.021182,0.150556,53.12,1200.0,67.97,0.59,8.21,59.08
4,2022,2100303,Aldeias Altas,MA,0.008863,0.982759,0.241379,0.043889,0.033889,0.919074,...,0.213457,0.359990,0.023668,0.165767,43.84,1200.0,73.89,0.59,6.68,70.61


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1781 entries, 0 to 1780
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   NU_ANO_CENSO                   1781 non-null   int64  
 1   CO_MUNICIPIO                   1781 non-null   int64  
 2   NO_MUNICIPIO                   1781 non-null   str    
 3   SG_UF                          1781 non-null   str    
 4   TAXA_CRE_INT                   1781 non-null   float64
 5   PROP_ESC_PUBLICA               1781 non-null   float64
 6   PROP_ESC_URBANA                1781 non-null   float64
 7   PROP_MAT_BRANCA                1781 non-null   float64
 8   PROP_MAT_PRETA                 1781 non-null   float64
 9   PROP_MAT_PARDA                 1781 non-null   float64
 10  PROP_MAT_INDIGENA              1781 non-null   float64
 11  COBERTURA_CRECHE_0_3           1781 non-null   float64
 12  ESCOLAS_CRECHE_POR_1000_CRIAN  1781 non-null   float64
 13 

In [4]:
for col in df.columns:
    print(col, ':', df[col].nunique())

NU_ANO_CENSO : 1
CO_MUNICIPIO : 1781
NO_MUNICIPIO : 1714
SG_UF : 9
TAXA_CRE_INT : 1083
PROP_ESC_PUBLICA : 440
PROP_ESC_URBANA : 626
PROP_MAT_BRANCA : 1781
PROP_MAT_PRETA : 1748
PROP_MAT_PARDA : 1769
PROP_MAT_INDIGENA : 1105
COBERTURA_CRECHE_0_3 : 1751
ESCOLAS_CRECHE_POR_1000_CRIAN : 1652
MEDIA_MAT_0_3_POR_ESCOLA : 42
LOG_POP_TOTAL : 1725
PIB_PC : 1780
LOG_PIB_PC : 1779
TAXA_OCUP_FEM : 1773
TAXA_OCUP_MASC : 1776
PROP_MAES_JOVENS : 1725
PROP_DOM_SEM_CONJUGE_MULHER : 1760
PERC_RENDA_OUTRAS_FONTES : 1387
RENDA_MEDIANA_RESPONSAVEL : 51
IDH_PROP_POBREZA_CRIANCAS : 1449
IDH_GINI : 37
IDH_EXPECT_ANOS_ESTUDO : 363
IDH_RAZAO_DEPENDENCIA : 1231


NU_ANO_CENSO  vai ser dropada por ser constante. CO_MUNICIPIO e NO_MUNICIPIO são IDs (Vou manter só o nome msm pra ajudar a plotar). SG_UF Vamos tentar fazer um OHE depois e ver se dá pra usar. 

Carregando dados de treino já normalizados (Média = 0, Desvio_Padrão = 1)

In [8]:
X = pd.read_parquet('../Datasets/splits/X_train_scaled.parquet')
y = pd.read_parquet('../Datasets/splits/y_train.parquet')

### Montando Funções de Seleção de Features

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFE, SequentialFeatureSelector
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

In [9]:
def rmse_score(y_true, y_pred):
    return -np.sqrt(mean_squared_error(y_true, y_pred))

In [10]:
def rfe_search(X, y, cv=None, random_state=42):    
    results = []

    for i in range(1, len(X.columns)+1):
        selector = RFE(
            estimator=Ridge(random_state=random_state),
            n_features_to_select=i
        )

        pipeline = Pipeline([
            ("selector", selector),
            ("model", Ridge(random_state=random_state))
        ])

        rmse_scores = cross_val_score(
            pipeline, 
            X,
            y, 
            cv=cv,
            scoring="neg_root_mean_squared_error",
            n_jobs=os.cpu_count()//2
        )

        r2_scores = cross_val_score(
            pipeline,
            X,
            y,
            cv=cv,
            scoring="r2",
            n_jobs=os.cpu_count()//2
        )

        selector.fit(X, y)

        selected_features = X.columns[selector.get_support()].to_list() # o método support_ de selector pega os nomes das colunas que foram selecionadas pelo RFE.

        results.append({
            "method": "RFE",
            "n_features": i,
            "selected_features": selected_features,
            "rmse_mean": -rmse_scores.mean(),
            "rmse_std": rmse_scores.std(),
            "r2_mean": r2_scores.mean(),
            "r2_std": r2_scores.std()
        })

    return pd.DataFrame(results)

In [11]:
def sfs_search(X, y, cv, direction="forward", random_state=42):
    results = []

    for i in range(1, len(X.columns)):
        selector = SequentialFeatureSelector(
            estimator=Ridge(random_state=random_state),
            n_features_to_select=i,
            direction=direction,
            scoring="neg_root_mean_squared_error",
            cv=cv,
            n_jobs=os.cpu_count()//2
        )

        pipeline = Pipeline([
            ("selector", selector),
            ("model", Ridge(random_state=random_state))
        ])

        rmse_scores = cross_val_score(
            pipeline,
            X,
            y,
            cv=cv,
            scoring="neg_root_mean_squared_error", 
            n_jobs=os.cpu_count()//2
        )

        r2_scores = cross_val_score(
            pipeline,
            X,
            y,
            cv=cv,
            scoring="r2",
            n_jobs=os.cpu_count()//2
        )

        selector.fit(X, y)

        selected_features = X.columns[selector.get_support()].to_list() 

        results.append({
            "method": f"RFS_{direction}",
            "n_features": i,
            "selected_features": selected_features,
            "rmse_mean": -rmse_scores.mean(),
            "rmse_std": rmse_scores.std(),
            "r2_mean": r2_scores.mean(),
            "r2_std": r2_scores.std()
        })

    return pd.DataFrame(results)

In [ ]:
def run_feature_selection(X, y): # Aqui já é esperado um X normalizado.
    X = X.select_dtypes(include=[np.number])

    cv = KFold(
        n_splits=5,
        shuffle=True, 
        random_state=42
    )

    print('Executando Recursive Feature Elimination...')
    rfe_results = rfe_search(X, y, cv)
    
    print('Executando Sequential Feature Selection...')
    sfs_results = sfs_search(X, y, cv)


    results = pd.concat([
        rfe_results,
        sfs_results
        ],
        ignore_index=True
    )

    results_df = results.sort_values(
        by=["rmse_mean", "r2_mean"],
        ascending=[True, False]
    )

    best_result = results_df.iloc[0]

    return results_df, best_result

In [19]:
results_df, best_result = run_feature_selection(X_scaled, y)

Executando Recursive Feature Elimination...
Executando Sequential Feature Selection...
Executando "Lasso Selection"...


/home/guilherme_fernandes/anaconda3/envs/ml_mest/lib/python3.14/site-packages/sklearn/feature_selection/_base.py:122: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/home/guilherme_fernandes/anaconda3/envs/ml_mest/lib/python3.14/site-packages/sklearn/feature_selection/_base.py:122: UserWarning: No features were selected: either the data is too noisy or the selection test too strict.
  warnings.warn(
/home/guilherme_fernandes/anaconda3/envs/ml_mest/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:490: FitFailedWarning: 
2 fits failed out of a total of 5.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceba

In [23]:
results_df.to_csv('../Results/feature_selection.csv')